# Run Testing Pipeline For All Hand Models

This notebook runs the same prediction/evaluation path used by `run_testing_pipeline.py` across every discovered hand-only experiment run. It uses the testing videos in `Dataset/testing`, covers all models and scenarios from `run_hand_all_experiments.py`, and keeps separate rows for each frame sampling run.

In [6]:
from __future__ import annotations

import json
import re
import traceback
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from training.config import ALLOWED_MODELS, ALLOWED_SCENARIOS, ExperimentConfig
from training.testing_pipeline import evaluate_prediction, load_label_map, predict_video, resolve_run_dir

pd.set_option("display.max_colwidth", 120)


## Settings

In [7]:
DATASET_DIR = Path("Dataset/testing")
ARTIFACTS_DIR = Path("artifacts")
EXPERIMENTS_DIR = ARTIFACTS_DIR / "experiments"
OUTPUT_DIR = ARTIFACTS_DIR / "testing_pipeline_all_models"

SCENARIOS = tuple(ALLOWED_SCENARIOS)
MODELS = tuple(ALLOWED_MODELS)
MODALITY = ExperimentConfig().modality
TOP_K = 5

# Leave as None to evaluate every video. Set to a small integer for a quick smoke test.
MAX_VIDEOS = None

# Leave empty to test every discovered run. Otherwise paste run directories or .weights.h5 paths here.
# Each path represents one model/scenario/frame-sample run.
MODEL_PATHS = [
    # "artifacts/experiments/si/cnn1d/hand/20260503T011439Z",
    # "artifacts/experiments/sd/transformer/hand/20260505T190431Z/best.weights.h5",
]

# Optional extra filter when MODEL_PATHS is empty. Example: {10, 20, 30}
FRAME_SAMPLE_COUNTS = None

# How to handle multiple runs with the same scenario/model/modality/frame_sample_count.
# Use "latest" for the newest run_id, or "best" for the highest summary metric.
RUN_SELECTION_STRATEGY = "best"
BEST_SUMMARY_METRIC = "f1_score"  # Options commonly available: "accuracy", "f1_score". Lower "loss" is handled automatically.

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Testing videos: {DATASET_DIR}")
print(f"Experiments: {EXPERIMENTS_DIR}")
print(f"Outputs: {OUTPUT_DIR}")


Testing videos: Dataset/testing
Experiments: artifacts/experiments
Outputs: artifacts/testing_pipeline_all_models


## Discover Videos And Experiment Runs

In [8]:
def normalize_label_from_video_name(video_path: Path) -> str | None:
    """Infer a gloss label from names like 123456-GO TO_JAIL.mp4 -> GOTOJAIL."""
    stem = video_path.stem
    if "-" not in stem:
        return None
    raw_label = stem.split("-", 1)[1]
    return re.sub(r"[\s_]+", "", raw_label).upper()


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def discover_videos(dataset_dir: Path, max_videos: int | None = None) -> list[Path]:
    videos = sorted(path for path in dataset_dir.glob("*.mp4") if path.is_file())
    if max_videos is not None:
        videos = videos[:max_videos]
    return videos


def run_row_from_path(model_path: Path) -> dict:
    run_dir = resolve_run_dir(model_path)
    config_path = run_dir / "config.json"
    config = read_json(config_path)
    summary_path = run_dir / "summary.json"
    summary = read_json(summary_path) if summary_path.exists() else {}
    metrics = summary.get("metrics", {}) if isinstance(summary.get("metrics", {}), dict) else {}
    weights_path = model_path if model_path.is_file() and model_path.name.endswith(".weights.h5") else None
    if weights_path is None:
        weights_path = run_dir / "best.weights.h5" if (run_dir / "best.weights.h5").exists() else run_dir / "final.weights.h5"
    return {
        "scenario": str(config.get("scenario", "")),
        "model": str(config.get("model", "")),
        "modality": str(config.get("modality", "")),
        "frame_sample_count": int(config.get("frame_sample_count", -1)),
        "run_id": run_dir.name,
        "run_dir": run_dir,
        "config_path": config_path,
        "summary_path": summary_path if summary_path.exists() else None,
        "weights_path": weights_path,
        "input_shape": tuple(config.get("input_shape", [])),
        "num_classes": config.get("num_classes"),
        "summary_accuracy": metrics.get("accuracy"),
        "summary_f1_score": metrics.get("f1_score"),
        "summary_loss": metrics.get("loss"),
    }


def discover_runs(
    experiments_dir: Path,
    scenarios: tuple[str, ...],
    models: tuple[str, ...],
    modality: str,
    frame_sample_counts: set[int] | None = None,
) -> pd.DataFrame:
    rows = []
    for scenario in scenarios:
        for model in models:
            model_dir = experiments_dir / scenario / model / modality
            for config_path in sorted(model_dir.glob("*/config.json")):
                run_dir = config_path.parent
                config = read_json(config_path)
                frame_sample_count = int(config.get("frame_sample_count", -1))
                if frame_sample_counts is not None and frame_sample_count not in frame_sample_counts:
                    continue
                rows.append(run_row_from_path(run_dir))
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["scenario", "model", "frame_sample_count", "run_id"]).reset_index(drop=True)


def runs_from_model_paths(model_paths: list[str | Path]) -> pd.DataFrame:
    rows = [run_row_from_path(Path(path)) for path in model_paths]
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["scenario", "model", "frame_sample_count", "run_id"]).reset_index(drop=True)


def select_preferred_runs(runs: pd.DataFrame, strategy: str, best_metric: str) -> pd.DataFrame:
    if runs.empty:
        return runs

    key_columns = ["scenario", "model", "modality", "frame_sample_count"]
    strategy = strategy.lower().strip()
    if strategy == "latest":
        selected = runs.sort_values(key_columns + ["run_id"]).groupby(key_columns, as_index=False, dropna=False).tail(1)
    elif strategy == "best":
        metric_column = f"summary_{best_metric}"
        if metric_column not in runs.columns:
            raise ValueError(f"Unknown summary metric: {best_metric!r}. Expected a column named {metric_column!r}.")
        ascending = best_metric != "loss"
        sort_columns = key_columns + [metric_column, "run_id"]
        selected = (
            runs.sort_values(sort_columns, ascending=[True, True, True, True, ascending, True], na_position="first")
            .groupby(key_columns, as_index=False, dropna=False)
            .tail(1)
        )
    else:
        raise ValueError("RUN_SELECTION_STRATEGY must be 'latest' or 'best'.")

    return selected.sort_values(key_columns).reset_index(drop=True)


selected_sample_counts = None if FRAME_SAMPLE_COUNTS is None else {int(value) for value in FRAME_SAMPLE_COUNTS}
videos = discover_videos(DATASET_DIR, MAX_VIDEOS)
all_runs_df = discover_runs(EXPERIMENTS_DIR, SCENARIOS, MODELS, MODALITY, selected_sample_counts)
candidate_runs_df = runs_from_model_paths(MODEL_PATHS) if MODEL_PATHS else all_runs_df.copy()
runs_df = select_preferred_runs(candidate_runs_df, RUN_SELECTION_STRATEGY, BEST_SUMMARY_METRIC)

print(f"Videos discovered: {len(videos)}")
print(f"Experiment runs discovered: {len(all_runs_df)}")
print(f"Experiment run candidates: {len(candidate_runs_df)}")
print(f"Experiment runs selected for testing: {len(runs_df)} ({RUN_SELECTION_STRATEGY}, metric={BEST_SUMMARY_METRIC})")

model_path_list_df = all_runs_df[["scenario", "model", "frame_sample_count", "run_id", "summary_accuracy", "summary_f1_score", "summary_loss", "run_dir", "weights_path"]].copy()
model_path_list_df["run_dir"] = model_path_list_df["run_dir"].astype(str)
model_path_list_df["weights_path"] = model_path_list_df["weights_path"].astype(str)
display(model_path_list_df)

print("Selected runs:")
display(runs_df)


Videos discovered: 55
Experiment runs discovered: 27
Experiment run candidates: 27
Experiment runs selected for testing: 24 (best, metric=f1_score)


,scenario,model,frame_sample_count,run_id,summary_accuracy,summary_f1_score,summary_loss,run_dir,weights_path
0,sd,cnn1d,10,20260502T172408Z,0.156634,1.386587e-01,4.911060,artifacts/experiments/sd/cnn1d/hand/20260502T172408Z,artifacts/experiments/sd/cnn1d/hand/20260502T172408Z/best.weights.h5
1,sd,cnn1d,20,20260503T085939Z,0.184288,1.689406e-01,4.579266,artifacts/experiments/sd/cnn1d/hand/20260503T085939Z,artifacts/experiments/sd/cnn1d/hand/20260503T085939Z/best.weights.h5
2,sd,cnn1d,30,20260504T175400Z,0.185927,1.713791e-01,4.606834,artifacts/experiments/sd/cnn1d/hand/20260504T175400Z,artifacts/experiments/sd/cnn1d/hand/20260504T175400Z/best.weights.h5
3,sd,gcn,10,20260503T000033Z,0.147983,1.348650e-01,4.918382,artifacts/experiments/sd/gcn/hand/20260503T000033Z,artifacts/experiments/sd/gcn/hand/20260503T000033Z/best.weights.h5
4,sd,gcn,20,20260503T180131Z,0.172480,1.604728e-01,4.522992,artifacts/experiments/sd/gcn/hand/20260503T180131Z,artifacts/experiments/sd/gcn/hand/20260503T180131Z/best.weights.h5
5,sd,gcn,30,20260504T191712Z,0.166196,1.545003e-01,4.571587,artifacts/experiments/sd/gcn/hand/20260504T191712Z,artifacts/experiments/sd/gcn/hand/20260504T191712Z/best.weights.h5
6,sd,lstm,10,20260502T182524Z,0.248581,2.312013e-01,3.891554,artifacts/experiments/sd/lstm/hand/20260502T182524Z,artifacts/experiments/sd/lstm/hand/20260502T182524Z/best.weights.h5
7,sd,lstm,20,20260503T100158Z,0.364144,3.499793e-01,3.083986,artifacts/experiments/sd/lstm/hand/20260503T100158Z,artifacts/experiments/sd/lstm/hand/20260503T100158Z/best.weights.h5
8,sd,lstm,30,20260504T180433Z,0.408068,3.949070e-01,2.805501,artifacts/experiments/sd/lstm/hand/20260504T180433Z,artifacts/experiments/sd/lstm/hand/20260504T180433Z/best.weights.h5
9,sd,transformer,10,20260502T220632Z,0.411377,4.055214e-01,2.972796,artifacts/experiments/sd/transformer/hand/20260502T220632Z,artifacts/experiments/sd/transformer/hand/20260502T220632Z/best.weights.h5


Selected runs:


,scenario,model,modality,frame_sample_count,run_id,run_dir,config_path,summary_path,weights_path,input_shape,num_classes,summary_accuracy,summary_f1_score,summary_loss
0,sd,cnn1d,hand,10,20260502T172408Z,artifacts/experiments/sd/cnn1d/hand/20260502T172408Z,artifacts/experiments/sd/cnn1d/hand/20260502T172408Z/config.json,artifacts/experiments/sd/cnn1d/hand/20260502T172408Z/summary.json,artifacts/experiments/sd/cnn1d/hand/20260502T172408Z/best.weights.h5,"(10, 210)",2731,0.156634,0.138659,4.911060
1,sd,cnn1d,hand,20,20260503T085939Z,artifacts/experiments/sd/cnn1d/hand/20260503T085939Z,artifacts/experiments/sd/cnn1d/hand/20260503T085939Z/config.json,artifacts/experiments/sd/cnn1d/hand/20260503T085939Z/summary.json,artifacts/experiments/sd/cnn1d/hand/20260503T085939Z/best.weights.h5,"(20, 210)",2731,0.184288,0.168941,4.579266
2,sd,cnn1d,hand,30,20260504T175400Z,artifacts/experiments/sd/cnn1d/hand/20260504T175400Z,artifacts/experiments/sd/cnn1d/hand/20260504T175400Z/config.json,artifacts/experiments/sd/cnn1d/hand/20260504T175400Z/summary.json,artifacts/experiments/sd/cnn1d/hand/20260504T175400Z/best.weights.h5,"(30, 210)",2731,0.185927,0.171379,4.606834
3,sd,gcn,hand,10,20260503T000033Z,artifacts/experiments/sd/gcn/hand/20260503T000033Z,artifacts/experiments/sd/gcn/hand/20260503T000033Z/config.json,artifacts/experiments/sd/gcn/hand/20260503T000033Z/summary.json,artifacts/experiments/sd/gcn/hand/20260503T000033Z/best.weights.h5,"(10, 210)",2731,0.147983,0.134865,4.918382
4,sd,gcn,hand,20,20260503T180131Z,artifacts/experiments/sd/gcn/hand/20260503T180131Z,artifacts/experiments/sd/gcn/hand/20260503T180131Z/config.json,artifacts/experiments/sd/gcn/hand/20260503T180131Z/summary.json,artifacts/experiments/sd/gcn/hand/20260503T180131Z/best.weights.h5,"(20, 210)",2731,0.172480,0.160473,4.522992
5,sd,gcn,hand,30,20260504T191712Z,artifacts/experiments/sd/gcn/hand/20260504T191712Z,artifacts/experiments/sd/gcn/hand/20260504T191712Z/config.json,artifacts/experiments/sd/gcn/hand/20260504T191712Z/summary.json,artifacts/experiments/sd/gcn/hand/20260504T191712Z/best.weights.h5,"(30, 210)",2731,0.166196,0.154500,4.571587
6,sd,lstm,hand,10,20260502T182524Z,artifacts/experiments/sd/lstm/hand/20260502T182524Z,artifacts/experiments/sd/lstm/hand/20260502T182524Z/config.json,artifacts/experiments/sd/lstm/hand/20260502T182524Z/summary.json,artifacts/experiments/sd/lstm/hand/20260502T182524Z/best.weights.h5,"(10, 210)",2731,0.248581,0.231201,3.891554
7,sd,lstm,hand,20,20260503T100158Z,artifacts/experiments/sd/lstm/hand/20260503T100158Z,artifacts/experiments/sd/lstm/hand/20260503T100158Z/config.json,artifacts/experiments/sd/lstm/hand/20260503T100158Z/summary.json,artifacts/experiments/sd/lstm/hand/20260503T100158Z/best.weights.h5,"(20, 210)",2731,0.364144,0.349979,3.083986
8,sd,lstm,hand,30,20260504T180433Z,artifacts/experiments/sd/lstm/hand/20260504T180433Z,artifacts/experiments/sd/lstm/hand/20260504T180433Z/config.json,artifacts/experiments/sd/lstm/hand/20260504T180433Z/summary.json,artifacts/experiments/sd/lstm/hand/20260504T180433Z/best.weights.h5,"(30, 210)",2731,0.408068,0.394907,2.805501
9,sd,transformer,hand,10,20260502T220632Z,artifacts/experiments/sd/transformer/hand/20260502T220632Z,artifacts/experiments/sd/transformer/hand/20260502T220632Z/config.json,artifacts/experiments/sd/transformer/hand/20260502T220632Z/summary.json,artifacts/experiments/sd/transformer/hand/20260502T220632Z/best.weights.h5,"(10, 210)",2731,0.411377,0.405521,2.972796


In [9]:
if not videos:
    raise FileNotFoundError(f"No .mp4 videos found in {DATASET_DIR}")
if runs_df.empty:
    raise FileNotFoundError(f"No experiment runs found under {EXPERIMENTS_DIR}")

video_df = pd.DataFrame(
    {
        "video": videos,
        "video_name": [path.name for path in videos],
        "expected_label": [normalize_label_from_video_name(path) for path in videos],
    }
)
display(video_df.head(20))


,video,video_name,expected_label
0,Dataset/testing/019809089337332608-HEAP.mp4,019809089337332608-HEAP.mp4,HEAP
1,Dataset/testing/04044336822636918-SNOB.mp4,04044336822636918-SNOB.mp4,SNOB
2,Dataset/testing/042428497095387874-TEAR RIP_2.mp4,042428497095387874-TEAR RIP_2.mp4,TEARRIP2
3,Dataset/testing/08357883202605598-BACK UP.mp4,08357883202605598-BACK UP.mp4,BACKUP
4,Dataset/testing/10197122928088187-GRENADE.mp4,10197122928088187-GRENADE.mp4,GRENADE
5,Dataset/testing/12448238042056992-SEQUENCE.mp4,12448238042056992-SEQUENCE.mp4,SEQUENCE
6,Dataset/testing/1461714441572386-GRIND.mp4,1461714441572386-GRIND.mp4,GRIND
7,Dataset/testing/16186397288837262-SUSHI.mp4,16186397288837262-SUSHI.mp4,SUSHI
8,Dataset/testing/16570823618121522-CASTLE.mp4,16570823618121522-CASTLE.mp4,CASTLE
9,Dataset/testing/18508791786058-GRADUATE1.mp4,18508791786058-GRADUATE1.mp4,GRADUATE1


## Run All Predictions

This cell can take a while because each row runs video landmark extraction and model inference. Results are checkpointed to JSONL as they finish, so a long run still leaves useful output if it is interrupted.

In [10]:
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
jsonl_path = OUTPUT_DIR / f"testing_pipeline_predictions_{timestamp}.jsonl"
csv_path = OUTPUT_DIR / f"testing_pipeline_predictions_{timestamp}.csv"
summary_path = OUTPUT_DIR / f"testing_pipeline_summary_{timestamp}.csv"

records = []
total = len(runs_df) * len(video_df)
completed = 0

with jsonl_path.open("w", encoding="utf-8") as jsonl_file:
    for run in runs_df.itertuples(index=False):
        print(
            f"\n=== {run.scenario} | {run.model} | {run.modality} | "
            f"frames={run.frame_sample_count} | run={run.run_id} ==="
        )
        for video in video_df.itertuples(index=False):
            completed += 1
            print(f"[{completed}/{total}] {video.video_name}")
            try:
                result = predict_video(video.video, run.run_dir, top_k=TOP_K)
                result = evaluate_prediction(result, video.expected_label)
                top_prediction = result["top_predictions"][0]
                record = {
                    "ok": True,
                    "scenario": result["scenario"],
                    "model": result["model"],
                    "modality": result["modality"],
                    "frame_sample_count": result["frame_sample_count"],
                    "run_id": run.run_id,
                    "run_dir": result["run_dir"],
                    "weights_path": result["weights_path"],
                    "video_path": result["video_path"],
                    "video_name": video.video_name,
                    "expected_label": result["expected_label"],
                    "prediction": top_prediction["label"],
                    "prediction_index": top_prediction["index"],
                    "confidence": top_prediction["confidence"],
                    "correct": result["correct"],
                    "top_predictions": result["top_predictions"],
                    "error": None,
                }
            except Exception as exc:
                record = {
                    "ok": False,
                    "scenario": run.scenario,
                    "model": run.model,
                    "modality": run.modality,
                    "frame_sample_count": run.frame_sample_count,
                    "run_id": run.run_id,
                    "run_dir": str(run.run_dir),
                    "weights_path": str(run.weights_path),
                    "video_path": str(video.video),
                    "video_name": video.video_name,
                    "expected_label": video.expected_label,
                    "prediction": None,
                    "prediction_index": None,
                    "confidence": None,
                    "correct": None,
                    "top_predictions": [],
                    "error": "".join(traceback.format_exception_only(type(exc), exc)).strip(),
                }
            records.append(record)
            jsonl_file.write(json.dumps(record, ensure_ascii=False) + "\n")
            jsonl_file.flush()

results_df = pd.DataFrame(records)
results_df.to_csv(csv_path, index=False)
print(f"\nSaved detailed JSONL: {jsonl_path}")
print(f"Saved detailed CSV: {csv_path}")
display(results_df.head())



=== sd | cnn1d | hand | frames=10 | run=20260502T172408Z ===
[1/1320] 019809089337332608-HEAP.mp4


W0000 00:00:1779272561.161636   83797 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779272561.168505   83800 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


[2/1320] 04044336822636918-SNOB.mp4


W0000 00:00:1779272561.985458   83859 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779272561.993619   83859 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


[3/1320] 042428497095387874-TEAR RIP_2.mp4


W0000 00:00:1779272562.543074   83898 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779272562.548391   83897 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


... (2,632 outputs left out)

[1320/1320] 9725091268498984-REQUEST.mp4


W0000 00:00:1779274004.846285  144670 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779274004.852683  144663 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.



Saved detailed JSONL: artifacts/testing_pipeline_all_models/testing_pipeline_predictions_20260520T102241Z.jsonl
Saved detailed CSV: artifacts/testing_pipeline_all_models/testing_pipeline_predictions_20260520T102241Z.csv


,ok,scenario,model,modality,frame_sample_count,run_id,run_dir,weights_path,video_path,video_name,expected_label,prediction,prediction_index,confidence,correct,top_predictions,error
0,True,sd,cnn1d,hand,10,20260502T172408Z,artifacts/experiments/sd/cnn1d/hand/20260502T172408Z,artifacts/experiments/sd/cnn1d/hand/20260502T172408Z/best.weights.h5,Dataset/testing/019809089337332608-HEAP.mp4,019809089337332608-HEAP.mp4,HEAP,CHILD,428,0.035151,False,"[{'rank': 1, 'label': 'CHILD', 'index': 428, 'confidence': 0.03515135124325752}, {'rank': 2, 'label': 'STAFF', 'inde...",None
1,True,sd,cnn1d,hand,10,20260502T172408Z,artifacts/experiments/sd/cnn1d/hand/20260502T172408Z,artifacts/experiments/sd/cnn1d/hand/20260502T172408Z/best.weights.h5,Dataset/testing/04044336822636918-SNOB.mp4,04044336822636918-SNOB.mp4,SNOB,SNOB,2179,0.192759,True,"[{'rank': 1, 'label': 'SNOB', 'index': 2179, 'confidence': 0.19275882840156555}, {'rank': 2, 'label': 'HUNDRED', 'in...",None
2,True,sd,cnn1d,hand,10,20260502T172408Z,artifacts/experiments/sd/cnn1d/hand/20260502T172408Z,artifacts/experiments/sd/cnn1d/hand/20260502T172408Z/best.weights.h5,Dataset/testing/042428497095387874-TEAR RIP_2.mp4,042428497095387874-TEAR RIP_2.mp4,TEARRIP2,HANGCLOTHES,1098,0.101827,False,"[{'rank': 1, 'label': 'HANGCLOTHES', 'index': 1098, 'confidence': 0.10182669758796692}, {'rank': 2, 'label': 'SUNDAY...",None
3,True,sd,cnn1d,hand,10,20260502T172408Z,artifacts/experiments/sd/cnn1d/hand/20260502T172408Z,artifacts/experiments/sd/cnn1d/hand/20260502T172408Z/best.weights.h5,Dataset/testing/08357883202605598-BACK UP.mp4,08357883202605598-BACK UP.mp4,BACKUP,TORNADO,2472,0.244532,False,"[{'rank': 1, 'label': 'TORNADO', 'index': 2472, 'confidence': 0.24453212320804596}, {'rank': 2, 'label': 'STRUCTURE'...",None
4,True,sd,cnn1d,hand,10,20260502T172408Z,artifacts/experiments/sd/cnn1d/hand/20260502T172408Z,artifacts/experiments/sd/cnn1d/hand/20260502T172408Z/best.weights.h5,Dataset/testing/10197122928088187-GRENADE.mp4,10197122928088187-GRENADE.mp4,GRENADE,INFECTION,1227,0.080731,False,"[{'rank': 1, 'label': 'INFECTION', 'index': 1227, 'confidence': 0.0807308480143547}, {'rank': 2, 'label': 'PIPE1', '...",None


## Summaries

In [14]:
if "results_df" not in globals():
    latest_jsonl = sorted(OUTPUT_DIR.glob("testing_pipeline_predictions_*.jsonl"))[-1]
    results_df = pd.read_json(latest_jsonl, lines=True)
    print(f"Loaded {latest_jsonl}")
    summary_path = OUTPUT_DIR / latest_jsonl.name.replace("testing_pipeline_predictions_", "testing_pipeline_summary_").replace(".jsonl", ".csv")


def macro_f1_score_from_group(group: pd.DataFrame) -> float:
    valid = group[group["ok"] & group["expected_label"].notna() & group["prediction"].notna()]
    if valid.empty:
        return np.nan

    labels = sorted(set(valid["expected_label"]) | set(valid["prediction"]))
    f1_scores = []
    for label in labels:
        y_true = valid["expected_label"] == label
        y_pred = valid["prediction"] == label
        true_positive = int((y_true & y_pred).sum())
        false_positive = int((~y_true & y_pred).sum())
        false_negative = int((y_true & ~y_pred).sum())
        precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) else 0.0
        recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        f1_scores.append(f1)
    return float(np.mean(f1_scores))


summary_df = (
    results_df.assign(correct_numeric=results_df["correct"].map({True: 1, False: 0}))
    .groupby(["scenario", "model", "modality", "frame_sample_count", "run_id"], dropna=False)
    .agg(
        videos=("video_name", "count"),
        successful=("ok", "sum"),
        errors=("ok", lambda values: int((~values).sum())),
        accuracy=("correct_numeric", "mean"),
        mean_confidence=("confidence", "mean"),
    )
    .reset_index()
    .sort_values(["scenario", "frame_sample_count", "accuracy", "model"], ascending=[True, True, False, True])
)
f1_df = (
    results_df.groupby(["scenario", "model", "modality", "frame_sample_count", "run_id"], dropna=False)
    .apply(macro_f1_score_from_group)
    .rename("macro_f1")
    .reset_index()
)
summary_df = summary_df.merge(f1_df, on=["scenario", "model", "modality", "frame_sample_count", "run_id"], how="left")

summary_df.to_csv(summary_path, index=False)
print(f"Saved summary CSV: {summary_path}")
display(summary_df)


Saved summary CSV: artifacts/testing_pipeline_all_models/testing_pipeline_summary_20260520T102241Z.csv


,scenario,model,modality,frame_sample_count,run_id,videos,successful,errors,accuracy,mean_confidence
9,sd,transformer,hand,10,20260502T220632Z,55,55,0,0.236364,0.463674
6,sd,lstm,hand,10,20260502T182524Z,55,55,0,0.145455,0.243588
0,sd,cnn1d,hand,10,20260502T172408Z,55,55,0,0.127273,0.222894
3,sd,gcn,hand,10,20260503T000033Z,55,55,0,0.127273,0.158033
10,sd,transformer,hand,20,20260505T190431Z,55,55,0,0.363636,0.546814
7,sd,lstm,hand,20,20260503T100158Z,55,55,0,0.290909,0.342427
4,sd,gcn,hand,20,20260503T180131Z,55,55,0,0.145455,0.219882
1,sd,cnn1d,hand,20,20260503T085939Z,55,55,0,0.090909,0.282190
11,sd,transformer,hand,30,20260505T190035Z,55,55,0,0.309091,0.465779
8,sd,lstm,hand,30,20260504T180433Z,55,55,0,0.236364,0.381243


## Testing Pipeline Accuracy And F1 Bar Charts

These charts use `summary_df`, so they reflect the current testing pipeline predictions over `Dataset/testing`.

In [ ]:
testing_metric_df = (
    summary_df.assign(
        scenario=lambda df: df["scenario"].str.upper(),
        accuracy=lambda df: pd.to_numeric(df["accuracy"], errors="coerce"),
        macro_f1=lambda df: pd.to_numeric(df["macro_f1"], errors="coerce"),
    )
    [["scenario", "model", "modality", "frame_sample_count", "run_id", "videos", "successful", "errors", "accuracy", "macro_f1", "mean_confidence"]]
    .sort_values(["model", "frame_sample_count", "scenario"])
    .reset_index(drop=True)
)

display(
    testing_metric_df.style.format(
        {"accuracy": "{:.4f}", "macro_f1": "{:.4f}", "mean_confidence": "{:.4f}"},
        na_rep="",
    )
)

sample_colors = {
    10: "#60a5fa",
    20: "#fb923c",
    30: "#4ade80",
}
scenario_hatches = {
    "SD": "",
    "SI": "//",
}


def plot_testing_metric_bars(metric_column: str, title: str, ylabel: str, output_name: str) -> None:
    model_order = [model for model in ["cnn1d", "gcn", "lstm", "transformer"] if model in set(testing_metric_df["model"])]
    sample_counts = sorted(testing_metric_df["frame_sample_count"].dropna().unique())
    scenario_order = [scenario for scenario in ["SD", "SI"] if scenario in set(testing_metric_df["scenario"])]

    fig, ax = plt.subplots(figsize=(22, 7))
    x_positions = list(range(len(model_order)))
    bars_per_model = max(1, len(sample_counts) * len(scenario_order))
    bar_width = min(0.14, 0.86 / bars_per_model)
    metric_max = 0.0

    for sample_index, sample_count in enumerate(sample_counts):
        for scenario_index, scenario in enumerate(scenario_order):
            metric_slice = (
                testing_metric_df[
                    (testing_metric_df["frame_sample_count"] == sample_count)
                    & (testing_metric_df["scenario"] == scenario)
                ]
                .set_index("model")
                .reindex(model_order)
            )
            values = metric_slice[metric_column].tolist()
            metric_max = max(metric_max, max([value for value in values if pd.notna(value)], default=0.0))
            slot_index = sample_index * len(scenario_order) + scenario_index
            offset = (slot_index - (bars_per_model - 1) / 2) * bar_width
            bars = ax.bar(
                [x + offset for x in x_positions],
                values,
                width=bar_width,
                color=sample_colors.get(int(sample_count), "#94a3b8"),
                edgecolor="#1f2937",
                linewidth=0.7,
                hatch=scenario_hatches.get(scenario, ""),
                label=f"{int(sample_count)} frames {scenario}",
                zorder=2,
            )
            labels = [f"{value:.4f}" if pd.notna(value) else "" for value in values]
            ax.bar_label(bars, labels=labels, padding=3, fontsize=10, color="#111827")

    ax.set_title(title, fontsize=16)
    ax.set_ylabel(ylabel, fontsize=15)
    ax.set_xlabel("Model", fontsize=15)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(model_order, fontsize=13)
    ax.tick_params(axis="y", labelsize=13)
    ax.set_ylim(0, max(0.05, min(1.0, metric_max * 1.2)))
    ax.grid(axis="y", alpha=0.3, zorder=1)
    ax.legend(title="Sample / scenario", ncol=3, fontsize=12)
    fig.tight_layout()

    output_path = OUTPUT_DIR / output_name
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    print(f"Saved {output_path}")
    plt.show()


plot_testing_metric_bars(
    "accuracy",
    "Testing Pipeline Accuracy by Frame Sample Count",
    "Accuracy on Dataset/testing",
    "testing_pipeline_accuracy_bar.png",
)
plot_testing_metric_bars(
    "macro_f1",
    "Testing Pipeline Macro F1 by Frame Sample Count",
    "Macro F1 on Dataset/testing",
    "testing_pipeline_macro_f1_bar.png",
)


In [15]:
error_df = results_df.loc[~results_df["ok"], ["scenario", "model", "frame_sample_count", "run_id", "video_name", "error"]]
print(f"Errors: {len(error_df)}")
display(error_df.head(50))


Errors: 0


,scenario,model,frame_sample_count,run_id,video_name,error


## Optional: Run One Case Through The CLI

Use this cell when you want to verify the notebook is invoking the same command-line script behavior for a single video/run pair.

In [ ]:
import subprocess
import sys

sample_video = video_df.iloc[0]
sample_run = runs_df.iloc[0]

command = [
    sys.executable,
    "run_testing_pipeline.py",
    str(sample_video["video"]),
    str(sample_run["run_dir"]),
    "--expected-label",
    str(sample_video["expected_label"]),
    "--top-k",
    str(TOP_K),
]

print(" ".join(command))
subprocess.run(command, check=True)
